In [1]:
import json
import os
import random
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import gc

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    SEED,
    MODEL_2,
    MAX_LENGTH,
    LEARNING_RATE,
    BATCH_SIZE,
    NUM_EPOCHS,
    WEIGHT_DECAY,
    LABEL_NAMES,
    LABEL2ID,
    ID2LABEL,
    set_seed,
)
from src.data_utils import load_jsonl, tokenize_and_align_labels, prepare_datasets
from src.metrics import compute_metrics
from src.models_utils import get_model

set_seed()

label_names = LABEL_NAMES
label2id = LABEL2ID
id2label = ID2LABEL

print("Model 2:", MODEL_2)
print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())


Model 2: FacebookAI/xlm-roberta-base
Seed: 42
CUDA available: True


In [2]:
tokenizer_2, train_2, dev_2, test_2 = prepare_datasets(MODEL_2)


Map:   0%|          | 0/27482 [00:00<?, ? examples/s]

Map:   0%|          | 0/4767 [00:00<?, ? examples/s]

Map:   0%|          | 0/4126 [00:00<?, ? examples/s]

In [3]:
print("Train:", len(train_2))
print("Dev:", len(dev_2))
print("Test:", len(test_2))

print(tokenizer_2)

Train: 27482
Dev: 4767
Test: 4126
XLMRobertaTokenizerFast(name_or_path='FacebookAI/xlm-roberta-base', vocab_size=250002, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	250001: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=False, special=True),
}


In [4]:
model_2 = get_model(
    MODEL_2,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

model_2

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLMRobertaForTokenClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_f

In [5]:
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

training_args_2 = TrainingArguments(
    output_dir="../models/xlmr-ner",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_strategy="epoch",
    fp16=True,
    gradient_checkpointing=False,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

In [6]:
data_collator_2 = DataCollatorForTokenClassification(
    tokenizer=tokenizer_2
)

trainer_2 = Trainer(
    model=model_2,
    args=training_args_2,
    train_dataset=train_2,
    eval_dataset=dev_2,
    data_collator=data_collator_2,
    processing_class=tokenizer_2,
    compute_metrics=compute_metrics,
)

In [7]:
gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated() / 1024**2)
print(torch.cuda.memory_reserved() / 1024**2)

1058.49560546875
1096.0


In [8]:
train_result_2 = trainer_2.train()

  0%|          | 0/82446 [00:00<?, ?it/s]

{'loss': 0.0872, 'grad_norm': 0.0008231330430135131, 'learning_rate': 1.3337457244742013e-05, 'epoch': 1.0}


  0%|          | 0/4767 [00:00<?, ?it/s]

{'eval_loss': 0.10060316324234009, 'eval_precision': 0.7898697008625436, 'eval_recall': 0.8248371023380605, 'eval_f1': 0.8069747820380613, 'eval_accuracy': 0.9813407110943977, 'eval_runtime': 25.2243, 'eval_samples_per_second': 188.985, 'eval_steps_per_second': 188.985, 'epoch': 1.0}
{'loss': 0.0459, 'grad_norm': 0.0009810967603698373, 'learning_rate': 6.672973825291707e-06, 'epoch': 2.0}


  0%|          | 0/4767 [00:00<?, ?it/s]

{'eval_loss': 0.086387500166893, 'eval_precision': 0.8304956488838441, 'eval_recall': 0.8413185128401687, 'eval_f1': 0.8358720487433359, 'eval_accuracy': 0.9854683177093126, 'eval_runtime': 24.9325, 'eval_samples_per_second': 191.196, 'eval_steps_per_second': 191.196, 'epoch': 2.0}
{'loss': 0.0248, 'grad_norm': 9.885711187962443e-05, 'learning_rate': 8.490405841399219e-09, 'epoch': 3.0}


  0%|          | 0/4767 [00:00<?, ?it/s]

{'eval_loss': 0.09960207343101501, 'eval_precision': 0.8567551851133941, 'eval_recall': 0.8470678420850901, 'eval_f1': 0.8518839741736534, 'eval_accuracy': 0.986202512378343, 'eval_runtime': 25.0144, 'eval_samples_per_second': 190.571, 'eval_steps_per_second': 190.571, 'epoch': 3.0}
{'train_runtime': 14596.8685, 'train_samples_per_second': 5.648, 'train_steps_per_second': 5.648, 'train_loss': 0.05263666589224879, 'epoch': 3.0}


In [9]:
MODEL_2_OUTPUT = "../models/xlmr-ner-best"

trainer_2.save_model(MODEL_2_OUTPUT)
tokenizer_2.save_pretrained(MODEL_2_OUTPUT)

print("Model saved to:", MODEL_2_OUTPUT)

Model saved to: ../models/xlmr-ner-best


In [10]:
dev_results_2 = trainer_2.evaluate()

print("XLM-RoBERTa — Development results")

for metric, value in dev_results_2.items():
    print(f"{metric}: {value}")

  0%|          | 0/4767 [00:00<?, ?it/s]

XLM-RoBERTa — Development results
eval_loss: 0.09960207343101501
eval_precision: 0.8567551851133941
eval_recall: 0.8470678420850901
eval_f1: 0.8518839741736534
eval_accuracy: 0.986202512378343
eval_runtime: 24.384
eval_samples_per_second: 195.497
eval_steps_per_second: 195.497
epoch: 3.0
